In [1]:
from pathlib import Path
import numpy as np
import rasterio
import re
from rasterio.warp import reproject, Resampling

_ROOT = Path("/Users/farhanhassan/makeathon-challenge-2026")
S1_DIR  = _ROOT / "data" / "concept_data" / "sentinel-1"
SLIM_DIR = _ROOT / "data" / "cleaned" / "features_slim"

NEW_BANDS = ["s1_max_drop", "s1_trend_slope", "s1_breakpoint_mag"]
print(f"Adding {len(NEW_BANDS)} bands: {NEW_BANDS}")

Adding 3 bands: ['s1_max_drop', 's1_trend_slope', 's1_breakpoint_mag']


In [2]:
def load_s1_timeseries(tile_id, split="train"):
    """
    Load monthly S1 backscatter as a time-ordered stack.
    Returns (stack, times, profile) where stack is (T, H, W) in dB,
    times is list of (year, month) tuples.
    Prefers descending orbit, falls back to ascending.
    """
    s1_dir = S1_DIR / split / f"{tile_id}__s1_rtc"
    if not s1_dir.exists():
        print(f"  ⚠️  No S1 dir: {s1_dir}")
        return None, None, None
    
    # Parse all files
    files = []
    for f in sorted(s1_dir.glob("*.tif")):
        m = re.match(r".+__s1_rtc_(\d{4})_(\d{1,2})_(ascending|descending)\.tif", f.name)
        if m:
            files.append((f, int(m.group(1)), int(m.group(2)), m.group(3)))
    
    # Prefer descending per month
    by_month = {}
    for f, yr, mo, orb in files:
        key = (yr, mo)
        if key not in by_month or orb == "descending":
            by_month[key] = f
    
    # Sort chronologically
    sorted_keys = sorted(by_month.keys())
    
    # Read first to get shape
    with rasterio.open(by_month[sorted_keys[0]]) as src:
        H, W = src.height, src.width
        profile = {'transform': src.transform, 'crs': src.crs, 'height': H, 'width': W}
    
    stack = np.full((len(sorted_keys), H, W), np.nan, dtype=np.float32)
    for i, key in enumerate(sorted_keys):
        with rasterio.open(by_month[key]) as src:
            data = src.read(1).astype(np.float32)
            # Convert linear to dB
            data = np.where(data > 0, 10 * np.log10(data), np.nan)
            if data.shape == (H, W):
                stack[i] = data
    
    print(f"  S1 timeseries: {len(sorted_keys)} months, {sorted_keys[0]} → {sorted_keys[-1]}")
    return stack, sorted_keys, profile

# Quick test
import geopandas as gpd
meta_dir = _ROOT / "data" / "concept_data" / "metadata"
train_tiles = gpd.read_file(meta_dir / "train_tiles.geojson")["name"].tolist()
test_tiles  = gpd.read_file(meta_dir / "test_tiles.geojson")["name"].tolist()

avail_train = [t for t in train_tiles if (SLIM_DIR / "train" / f"{t}_slim.tif").exists()]
avail_test  = [t for t in test_tiles  if (SLIM_DIR / "test"  / f"{t}_slim.tif").exists()]
print(f"Tiles with slim files: {len(avail_train)} train, {len(avail_test)} test")

stack, times, s1_prof = load_s1_timeseries(avail_train[0])
print(f"  Stack shape: {stack.shape}")

Tiles with slim files: 5 train, 5 test
  S1 timeseries: 70 months, (2020, 1) → (2025, 12)
  Stack shape: (70, 335, 335)


In [3]:
def compute_s1_temporal_features(stack, times):
    """
    From a (T, H, W) monthly dB stack, compute 3 features per pixel:
    - max_drop: largest month-to-month dB decrease (most negative diff)
    - trend_slope: linear regression slope over all months
    - breakpoint_mag: magnitude of change at the single biggest cumsum deviation
    """
    T, H, W = stack.shape
    flat = stack.reshape(T, -1)  # (T, N)
    N = flat.shape[1]
    
    # 1. Max drop: min of consecutive differences
    diffs = np.diff(flat, axis=0)  # (T-1, N)
    max_drop = np.nanmin(diffs, axis=0)  # most negative = biggest drop
    
    # 2. Trend slope: linear fit over time index
    t_idx = np.arange(T, dtype=np.float32)
    t_mean = t_idx.mean()
    t_var = ((t_idx - t_mean) ** 2).sum()
    # Vectorized slope: cov(t, x) / var(t)
    x_mean = np.nanmean(flat, axis=0)  # (N,)
    # Handle NaN: use nanmean-based approach
    numerator = np.zeros(N, dtype=np.float32)
    counts = np.zeros(N, dtype=np.float32)
    for i in range(T):
        valid = np.isfinite(flat[i])
        numerator[valid] += (t_idx[i] - t_mean) * (flat[i, valid] - x_mean[valid])
        counts[valid] += (t_idx[i] - t_mean) ** 2
    trend_slope = np.where(counts > 0, numerator / counts, 0.0)
    
    # 3. Breakpoint magnitude: CUSUM-based
    # Cumulative sum of deviations from overall mean
    deviations = flat - x_mean[np.newaxis, :]  # (T, N)
    deviations = np.where(np.isfinite(deviations), deviations, 0.0)
    cusum = np.cumsum(deviations, axis=0)  # (T, N)
    # Breakpoint = month of max absolute CUSUM
    bp_idx = np.argmax(np.abs(cusum), axis=0)  # (N,)
    # Magnitude = mean_after_bp - mean_before_bp
    breakpoint_mag = np.full(N, np.nan, dtype=np.float32)
    for px in range(N):
        bp = bp_idx[px]
        if bp < 2 or bp > T - 3:
            breakpoint_mag[px] = 0.0
            continue
        before = flat[:bp, px]
        after = flat[bp:, px]
        b_mean = np.nanmean(before) if np.any(np.isfinite(before)) else 0
        a_mean = np.nanmean(after) if np.any(np.isfinite(after)) else 0
        breakpoint_mag[px] = a_mean - b_mean
    
    return {
        's1_max_drop': max_drop.reshape(H, W),
        's1_trend_slope': trend_slope.reshape(H, W),
        's1_breakpoint_mag': breakpoint_mag.reshape(H, W),
    }

# Test on first tile
feats = compute_s1_temporal_features(stack, times)
for k, v in feats.items():
    valid = v[np.isfinite(v)]
    print(f"  {k}: mean={valid.mean():.4f}, std={valid.std():.4f}, range=[{valid.min():.3f}, {valid.max():.3f}]")

  s1_max_drop: mean=-5.2581, std=1.0559, range=[-13.935, -2.239]
  s1_trend_slope: mean=-0.0030, std=0.0197, range=[-0.168, 0.143]
  s1_breakpoint_mag: mean=-0.1725, std=1.2120, range=[-8.927, 7.718]


In [4]:
def add_bands_to_slim(tile_id, split, s1_feats, s1_profile):
    """
    Read existing slim file, reproject new S1 bands to its grid, 
    write new file with 16 bands.
    """
    slim_path = SLIM_DIR / split / f"{tile_id}_slim.tif"
    
    with rasterio.open(slim_path) as src:
        old_data = src.read()          # (13, H, W)
        old_descs = list(src.descriptions)
        profile = src.profile.copy()
        dst_tf = src.transform
        dst_crs = src.crs
        H, W = src.height, src.width
    
    # Reproject each new S1 band to AEF/slim grid
    new_bands = []
    new_names = []
    for name in NEW_BANDS:
        reprojected = np.full((H, W), np.nan, dtype=np.float32)
        reproject(
            source=s1_feats[name],
            destination=reprojected,
            src_transform=s1_profile['transform'],
            src_crs=s1_profile['crs'],
            dst_transform=dst_tf,
            dst_crs=dst_crs,
            resampling=Resampling.bilinear,
        )
        new_bands.append(reprojected)
        new_names.append(name)
    
    # Stack old + new
    combined = np.concatenate([old_data, np.array(new_bands)], axis=0)
    all_descs = old_descs + new_names
    
    # Write
    profile.update(count=combined.shape[0], compress="deflate", predictor=2)
    with rasterio.open(slim_path, "w", **profile) as dst:
        dst.write(combined)
        dst.descriptions = tuple(all_descs)
    
    mb = slim_path.stat().st_size / 1e6
    print(f"  ✅ {tile_id} ({split}): {combined.shape[0]} bands, {mb:.1f} MB")

print("Function defined.")

Function defined.


In [5]:
# Process all tiles
for split, tile_list in [("train", avail_train), ("test", avail_test)]:
    print(f"\n=== {split.upper()} ===")
    for tile in tile_list:
        print(f"\n  {tile}:")
        stack, times, s1_prof = load_s1_timeseries(tile, split)
        if stack is None:
            print("    Skipped (no S1 data)")
            continue
        feats = compute_s1_temporal_features(stack, times)
        add_bands_to_slim(tile, split, feats, s1_prof)

print("\nDone!")


=== TRAIN ===

  18NXH_6_8:
  S1 timeseries: 70 months, (2020, 1) → (2025, 12)
  ✅ 18NXH_6_8 (train): 19 bands, 24.5 MB

  18NWG_6_6:
  S1 timeseries: 70 months, (2020, 1) → (2025, 12)
  ✅ 18NWG_6_6 (train): 19 bands, 55.7 MB

  18NWJ_8_9:
  S1 timeseries: 70 months, (2020, 1) → (2025, 12)
  ✅ 18NWJ_8_9 (train): 19 bands, 56.3 MB

  18NWH_1_4:
  S1 timeseries: 70 months, (2020, 1) → (2025, 12)


/var/folders/x5/_0qtdzg57193jjdlt702pxq80000gp/T/ipykernel_70094/3173440551.py:14: RuntimeWarning: All-NaN slice encountered
  max_drop = np.nanmin(diffs, axis=0)  # most negative = biggest drop
/var/folders/x5/_0qtdzg57193jjdlt702pxq80000gp/T/ipykernel_70094/3173440551.py:21: RuntimeWarning: Mean of empty slice
  x_mean = np.nanmean(flat, axis=0)  # (N,)
/var/folders/x5/_0qtdzg57193jjdlt702pxq80000gp/T/ipykernel_70094/3173440551.py:29: RuntimeWarning: invalid value encountered in divide
  trend_slope = np.where(counts > 0, numerator / counts, 0.0)


  ✅ 18NWH_1_4 (train): 19 bands, 56.3 MB

  18NWM_9_4:
  S1 timeseries: 70 months, (2020, 1) → (2025, 12)


/var/folders/x5/_0qtdzg57193jjdlt702pxq80000gp/T/ipykernel_70094/3173440551.py:14: RuntimeWarning: All-NaN slice encountered
  max_drop = np.nanmin(diffs, axis=0)  # most negative = biggest drop
/var/folders/x5/_0qtdzg57193jjdlt702pxq80000gp/T/ipykernel_70094/3173440551.py:21: RuntimeWarning: Mean of empty slice
  x_mean = np.nanmean(flat, axis=0)  # (N,)
/var/folders/x5/_0qtdzg57193jjdlt702pxq80000gp/T/ipykernel_70094/3173440551.py:29: RuntimeWarning: invalid value encountered in divide
  trend_slope = np.where(counts > 0, numerator / counts, 0.0)


  ✅ 18NWM_9_4 (train): 19 bands, 56.8 MB

=== TEST ===

  18NVJ_1_6:
  S1 timeseries: 72 months, (2020, 1) → (2025, 12)


...

  ✅ 33NTE_5_1 (test): 19 bands, 56.4 MB

  47QMA_6_2:
  S1 timeseries: 72 months, (2020, 1) → (2025, 12)


/var/folders/x5/_0qtdzg57193jjdlt702pxq80000gp/T/ipykernel_70094/3173440551.py:14: RuntimeWarning: All-NaN slice encountered
  max_drop = np.nanmin(diffs, axis=0)  # most negative = biggest drop
/var/folders/x5/_0qtdzg57193jjdlt702pxq80000gp/T/ipykernel_70094/3173440551.py:21: RuntimeWarning: Mean of empty slice
  x_mean = np.nanmean(flat, axis=0)  # (N,)
/var/folders/x5/_0qtdzg57193jjdlt702pxq80000gp/T/ipykernel_70094/3173440551.py:29: RuntimeWarning: invalid value encountered in divide
  trend_slope = np.where(counts > 0, numerator / counts, 0.0)


  ✅ 47QMA_6_2 (test): 19 bands, 46.5 MB

  48PWA_0_6:
  S1 timeseries: 72 months, (2020, 1) → (2025, 12)


/var/folders/x5/_0qtdzg57193jjdlt702pxq80000gp/T/ipykernel_70094/3173440551.py:14: RuntimeWarning: All-NaN slice encountered
  max_drop = np.nanmin(diffs, axis=0)  # most negative = biggest drop
/var/folders/x5/_0qtdzg57193jjdlt702pxq80000gp/T/ipykernel_70094/3173440551.py:21: RuntimeWarning: Mean of empty slice
  x_mean = np.nanmean(flat, axis=0)  # (N,)
/var/folders/x5/_0qtdzg57193jjdlt702pxq80000gp/T/ipykernel_70094/3173440551.py:29: RuntimeWarning: invalid value encountered in divide
  trend_slope = np.where(counts > 0, numerator / counts, 0.0)


  ✅ 48PWA_0_6 (test): 19 bands, 51.7 MB

Done!


In [6]:
# Sanity check
sample = list((SLIM_DIR / "train").glob("*_slim.tif"))[0]
with rasterio.open(sample) as src:
    print(f"File: {sample.name}")
    print(f"Bands: {src.count}")
    print(f"Names: {list(src.descriptions)}")
    data = src.read()
    print(f"Shape: {data.shape}, dtype: {data.dtype}")
    print(f"File size: {sample.stat().st_size / 1e6:.1f} MB")
    # Stats for new bands only
    for i, name in enumerate(src.descriptions):
        if name and name.startswith("s1_"):
            band = data[i]
            v = band[np.isfinite(band)]
            print(f"  {name}: mean={v.mean():.4f}, std={v.std():.4f}")

total = sum(f.stat().st_size for f in SLIM_DIR.rglob("*_slim.tif"))
print(f"\nTotal disk: {total / 1e6:.0f} MB")

File: 18NWG_6_6_slim.tif
Bands: 19
Names: ['aef_cosine_dist_2021', 'aef_cosine_dist_2022', 'aef_cosine_dist_2023', 'aef_cosine_dist_2024', 'aef_cosine_dist_2025', 'aef_delta_norm_2021', 'aef_delta_norm_2022', 'aef_delta_norm_2023', 'aef_delta_norm_2024', 'aef_delta_norm_2025', 's1_change_mean', 's1_change_ratio', 's1_after_std', 's1_max_drop', 's1_trend_slope', 's1_breakpoint_mag', 's1_max_drop', 's1_trend_slope', 's1_breakpoint_mag']
Shape: (19, 1004, 998), dtype: float32
File size: 55.7 MB
  s1_change_mean: mean=-0.8194, std=1.0062
  s1_change_ratio: mean=1.1410, std=0.1713
  s1_after_std: mean=1.6489, std=0.2289
  s1_max_drop: mean=-5.2891, std=0.7844
  s1_trend_slope: mean=-0.0151, std=0.0223
  s1_breakpoint_mag: mean=-0.7824, std=1.2023
  s1_max_drop: mean=-5.2891, std=0.7844
  s1_trend_slope: mean=-0.0151, std=0.0223
  s1_breakpoint_mag: mean=-0.7824, std=1.2023

Total disk: 515 MB


In [9]:
# Fix: slim files have 19 bands (3 temporal bands duplicated). Strip to 16.
for split in ["train", "test"]:
    d = SLIM_DIR / split
    for f in sorted(d.glob("*_slim.tif")):
        with rasterio.open(f) as src:
            if src.count <= 16:
                continue
            data = src.read()[:16]  # keep first 16 only
            descs = list(src.descriptions)[:16]
            profile = src.profile.copy()
        profile.update(count=16)
        with rasterio.open(f, "w", **profile) as dst:
            dst.write(data)
            dst.descriptions = tuple(descs)
        print(f"  Fixed {f.name}: {src.count} → 16 bands")

# Verify
sample = list((SLIM_DIR / "train").glob("*_slim.tif"))[0]
with rasterio.open(sample) as src:
    print(f"\nVerify: {sample.name} → {src.count} bands: {list(src.descriptions)}")

  Fixed 18NWG_6_6_slim.tif: 19 → 16 bands
  Fixed 18NWH_1_4_slim.tif: 19 → 16 bands
  Fixed 18NWJ_8_9_slim.tif: 19 → 16 bands
  Fixed 18NWM_9_4_slim.tif: 19 → 16 bands
  Fixed 18NXH_6_8_slim.tif: 19 → 16 bands
  Fixed 18NVJ_1_6_slim.tif: 19 → 16 bands
  Fixed 18NYH_2_1_slim.tif: 19 → 16 bands
  Fixed 33NTE_5_1_slim.tif: 19 → 16 bands
  Fixed 47QMA_6_2_slim.tif: 19 → 16 bands
  Fixed 48PWA_0_6_slim.tif: 19 → 16 bands

Verify: 18NWG_6_6_slim.tif → 16 bands: ['aef_cosine_dist_2021', 'aef_cosine_dist_2022', 'aef_cosine_dist_2023', 'aef_cosine_dist_2024', 'aef_cosine_dist_2025', 'aef_delta_norm_2021', 'aef_delta_norm_2022', 'aef_delta_norm_2023', 'aef_delta_norm_2024', 'aef_delta_norm_2025', 's1_change_mean', 's1_change_ratio', 's1_after_std', 's1_max_drop', 's1_trend_slope', 's1_breakpoint_mag']


In [10]:
# AUC test for new features
from scipy.stats import mannwhitneyu

LABEL_DIR = _ROOT / "data" / "concept_data" / "labels" / "train"
rng = np.random.default_rng(42)

# Load ALL 16 bands + labels in one pass
all_bands = {}
all_labels = []

for tile in avail_train:
    with rasterio.open(SLIM_DIR / "train" / f"{tile}_slim.tif") as src:
        descs = list(src.descriptions)
        H, W = src.height, src.width
        ft, fc = src.transform, src.crs
        for i, d in enumerate(descs):
            if d not in all_bands:
                all_bands[d] = []
            all_bands[d].append(src.read(i + 1).ravel())
    
    radd = np.zeros((H, W), dtype=np.int32)
    with rasterio.open(LABEL_DIR / "radd" / f"radd_{tile}_labels.tif") as lsrc:
        reproject(lsrc.read(1).astype(np.int32), radd,
                  src_transform=lsrc.transform, src_crs=lsrc.crs,
                  dst_transform=ft, dst_crs=fc, resampling=Resampling.nearest)
    all_labels.append((radd > 0).astype(np.int32).ravel())

labels = np.concatenate(all_labels)
all_bands = {k: np.concatenate(v) for k, v in all_bands.items()}
band_names = list(all_bands.keys())

# Valid = finite across ALL 16 bands
valid = np.ones(len(labels), dtype=bool)
for v in all_bands.values():
    valid &= np.isfinite(v)
labels = labels[valid]
all_bands = {k: v[valid] for k, v in all_bands.items()}

print(f"Valid pixels: {len(labels):,}  Defor: {(labels==1).sum():,}")

# AUC for new bands
n = min(10_000, (labels==1).sum(), (labels==0).sum())
d_idx = rng.choice(np.where(labels==1)[0], n, replace=False)
s_idx = rng.choice(np.where(labels==0)[0], n, replace=False)

print(f"\n{'Feature':<25} {'Defor μ':>9} {'Stable μ':>9} {'AUC':>6}")
print("-" * 52)
for name in NEW_BANDS:
    d, s = all_bands[name][d_idx], all_bands[name][s_idx]
    u, _ = mannwhitneyu(d, s, alternative='two-sided')
    auc = max(u/(n*n), 1-u/(n*n))
    print(f"{name:<25} {d.mean():>9.4f} {s.mean():>9.4f} {auc:>6.3f}")

# Compare: 13-band vs 16-band logistic regression
print("\n=== Logistic Regression: 13-band vs 16-band ===")
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.model_selection import cross_val_predict

n_sub = min(100_000, len(labels))
idx = rng.choice(len(labels), n_sub, replace=False)
X16 = np.column_stack([all_bands[b][idx] for b in band_names])
X13 = X16[:, :13]
y = labels[idx]

# Replace any remaining NaN with 0 (safety)
X16 = np.nan_to_num(X16, nan=0.0)
X13 = np.nan_to_num(X13, nan=0.0)

for label, X in [("13-band (old)", X13), ("16-band (new)", X16)]:
    lr = LogisticRegression(max_iter=500, class_weight='balanced', random_state=42)
    yp = cross_val_predict(lr, X, y, cv=5, method='predict_proba')[:, 1]
    print(f"  {label}: AUC={roc_auc_score(y, yp):.4f}  F1={f1_score(y, (yp>0.5).astype(int)):.3f}")

Valid pixels: 3,997,115  Defor: 494,423

Feature                     Defor μ  Stable μ    AUC
----------------------------------------------------
s1_max_drop                 -5.3934   -5.2009  0.586
s1_trend_slope              -0.0202   -0.0040  0.731
s1_breakpoint_mag           -1.0622   -0.2002  0.730

=== Logistic Regression: 13-band vs 16-band ===
  13-band (old): AUC=0.9499  F1=0.681
  16-band (new): AUC=0.9499  F1=0.680
